<a href="https://colab.research.google.com/github/JHoffmann12/JHMicroMagnetics/blob/main/mumax3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mumax3 in google colaboratory**

# About Google colaboratory

Google colaboratory is a research tool mainly used by researchers in the field of machine learning. The main purpose of this tool is to run python code in jupyter notebooks. These notebooks run on a virtual linux machine private to your gmail account. This means that you will need a gmail account to execute programs in a google colaboratory session. If you have a gmail account you should be able to copy this jupyter notebook to your google drive and execute the code cells.

To run mumax3 simulations, you do not need to write any python code. So you might wonder how we can use this jupyter notebook environment to run mumax3 simulations. The trick here is that you can execute shell commands by typing an exclamation mark followed by the command, as shown in the code cell below. If you run this code cell, the shell will print out the operating system of the virtual machine. Having the ability to run shell commands suffices to install and run mumax3 simulations, as demonstrated in the sections below.

In [ ]:
! echo "This machine runs" $(uname)

# Installing mumax3

To install mumax3.12 on this virtual machine, you can run the cell below. This might take a few minutes. When the installation is done, you can collapse this section to get a clean workspace.

In [ ]:
try:
    import google.colab
except ImportError:
    pass
else:
    # Download the mumax3 binary
    !wget https://mumax.ugent.be/mumax3-binaries/mumax3.12_linux_cuda12.9.tar.gz
    !tar -xvf mumax3.12_linux_cuda12.9.tar.gz
    !rm mumax3.12_linux_cuda12.9.tar.gz
    !rm -rf mumax3.12 && mv mumax3.12_linux_cuda12.9 mumax3.12
    #update the PATH environment variable
    import os
    os.environ['PATH'] += ":/content/mumax3.12"
    # Download an examplary script
    !wget https://raw.githubusercontent.com/JeroenMulkers/mumax3-tutorial/master/standardproblem4.mx3 -O standardproblem4.mx3

# Running a mumax3 script

You can open the filebrowser for this virtual machine on the left side of this page. Here you should see a mumax3 script named standardproblem4.mx3. You should be able to open this file with a double click. To execute the script run the code cell below.

In [ ]:
!mumax3 nonReciprocal2.mx3

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/nonReciprocal2', 'zip', '/content/nonReciprocal2.out')
files.download('/content/nonReciprocal2.zip')

In [10]:
!mumax3 MeronKinkMumax.mx3

//mumax 3.12 [linux_amd64 go1.22.4(gc) CUDA-12.9]
//commit hash: 6e5c98bb
//CPU info: Intel(R) Xeon(R) CPU @ 2.00GHz, Cores: 1, MHz: 2000.166
//GPU info: Tesla T4(14912MB), CUDA Driver 13.0, cc=7.5, using cc=75 PTX
//OS  info: Ubuntu 22.04.5 LTS, Hostname: 8195ce38507d
//Timestamp: 2026-09-03 21:27:17
//(c) Arne Vansteenkiste, Dynamat LAB, Ghent University, Belgium
//This is free software without any warranty. See license.txt
//********************************************************************//
//  If you use mumax in any work or publication,                      //
//  we kindly ask you to cite the references in references.bib        //
//********************************************************************//
//output directory: MeronKinkMumax.out/
//starting GUI at http://127.0.0.1:35367
Nx := 600
Ny := 440
Nz := 1
sizeX := 625e-12
sizeY := 625e-12
sizeZ := 625e-12
SetGridSize(Nx, Ny, Nz)
SetCellSize(sizeX, sizeY, sizeZ)
// WARNING: y-axis is not 7-smooth. It has 440 cells, with pr

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
import shutil
from google.colab import files

shutil.make_archive('/content/MeronKinkMumax', 'zip', '/content/MeronKinkMumax.out')
files.download('/content/MeronKinkMumax.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import glob, re, struct
import numpy as np, pandas as pd

def read_ovf(path):
    raw = open(path, 'rb').read()
    i = raw.find(b"# Begin: Data Binary 4\n")
    hdr = raw[:i].decode('latin1')
    meta = dict(re.findall(r"#\s*([^:\n]+):\s*(.*)", hdr))
    nx, ny, nz = (int(meta[k]) for k in ('xnodes','ynodes','znodes'))
    off = i + len(b"# Begin: Data Binary 4\n")
    assert abs(struct.unpack("<f", raw[off:off+4])[0] - 1234567.0) < 1e-3
    d = np.frombuffer(raw, dtype='<f4', count=nx*ny*nz*3, offset=off+4)
    d = np.moveaxis(d.reshape(nz, ny, nx, 3), 3, 0)      # components interleaved per cell
    t = float(re.search(r"time:\s*([0-9.eE+-]+)", hdr).group(1))
    return t, float(meta['xstepsize']), d

rows = []
for p in sorted(glob.glob('/content/MeronKinkMumax.out/m??????.ovf')):
    t, dx, d = read_ovf(p)
    mz, mx = d[2,0], d[0,0]
    w = 1.0 - mz**2                                       # sin^2(theta): weights the wall
    P = (w*mx).sum(0) / np.maximum(w.sum(0), 1e-30)       # wall phase, m_x not m_y
    s = np.sign(P)
    k = np.flatnonzero(s[:-1]*s[1:] < 0)
    if len(k) != 1:                                       # 0 or >1 meron -> run has broken down
        continue
    j = k[0]
    rows.append((t*1e9, (j + 0.5 + P[j]/(P[j]-P[j+1]))*dx*1e9))

pd.DataFrame(rows, columns=['time','x_coord']).to_csv('/content/traj_35mT.csv', index=False)
files.download('/content/traj_35mT.csv')